# RQFormer-2 Kaggle Full Workflow

Notebook này chạy trực tiếp toàn bộ workflow trên Kaggle, **không phụ thuộc `tools/run_all_rqformer_visualize.sh`**.

Pipeline:

1. Clone source từ GitHub hoặc fallback từ Kaggle input.
2. Link dataset đúng cấu trúc config gốc.
3. Patch config `_base_` từ `mmrotate::...` sang local base config để tránh lỗi `.mim/model-index.yml`.
4. Setup môi trường OpenMMLab/MMRotate.
5. Train hoặc resume từng dataset.
6. Test và lưu `predictions.pkl`.
7. Sinh chart/confusion nếu script/log có đủ.
8. Sinh bbox visualization và heatmap attention mẫu.
9. Ghi `summary_results.md`.
10. Zip output để tải về hoặc dùng resume.


## 0. Fine-tuning có phù hợp không?

Có. Source hiện tại vẫn dùng pipeline train/test gốc (`tools/train.py`, `tools/test.py`). Các module cải tiến nằm trong decoder/RRoI attention, có tham số học được và đi qua optimizer như layer bình thường.

Lưu ý:

- Không có checkpoint `.pth` vẫn train được từ đầu, nhưng lâu hơn.
- Nếu có checkpoint gốc, fine-tune nhanh hơn; các layer mới có thể báo `missing keys`, đây là bình thường.
- Chạy từng dataset nếu GPU time ít: `DATASETS="dior"`, `"dotav1_0"`, hoặc `"dotav1_5"`.
- Không visualize toàn bộ dataset; mặc định chỉ xuất `SAMPLE_IMAGES=10`.


## 1. Cấu hình Kaggle

Chỉnh cell này rồi bấm **Run All**.


In [3]:
from pathlib import Path

# ===== Git source =====
REPO_GIT_URL = "https://github.com/hungnp-dev/RQFormer.git"
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"  # nếu repo private
REPO_DIR = Path("/kaggle/working/RQFormer")

# Fallback nếu clone GitHub lỗi: upload source repo thành Kaggle Dataset tại path này.
REPO_INPUT_FALLBACK = Path("/kaggle/input/rqformer-source/RQFormer")

# ===== Dataset path =====
DATA_ROOT = Path("/kaggle/input/datasets/jks010/rqformer-dataset/data")
DATA_INPUTS = {
    "dior": DATA_ROOT / "DIOR",
    "dotav1_0": DATA_ROOT / "split_ss_dota",
    "dotav1_5": DATA_ROOT / "split_ss_dota1_5",
}

# ===== Optional checkpoint path =====
CKPT_INPUT = None  # Kh?ng d?ng checkpoint ngo?i; train t? ??u ho?c resume t? work_dirs
CKPT_DIR = Path("/kaggle/working/pth")

# ===== Output =====
WORK_ROOT = Path("/kaggle/working/work_dirs")

# ===== Dataset selection =====
DATASETS = "dior"  # train DIOR-R tr??c ?? l?y k?t qu?/visualization ?n ??nh

# ===== Runtime controls =====
DEVICE = "0,1"
NUM_GPUS = 2      # Kaggle dual T4; single GPU th? ??i 1
BATCH_SIZE = 1    # batch per GPU. Global batch=2, ?u ti?n ?n ??nh tr?nh grad_norm inf
NUM_WORKERS = 4   # ??c data nhanh h?n tr?n Kaggle
SAMPLE_IMAGES = 10  # g?n, ?? bbox/heatmap m?u
SCORE_THR = 0.5   # visualization s?ch h?n; eval mAP v?n theo evaluator/config
FORCE = 0
MAX_EPOCHS = 18   # train d?i h?n 6/12 ?? ch? s? v? visualization ??p h?n, v?n ng?n h?n full 36
LR = 1.25e-5      # scale LR theo global batch=2, ?n ??nh h?n LR g?c khi batch nh?
MASTER_PORT = 29500

# ===== Workflow stages =====
RUN_SETUP = 1
RUN_TRAIN = 1
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
RUN_SUMMARY = 1

print("REPO_GIT_URL =", REPO_GIT_URL)
print("DATA_ROOT =", DATA_ROOT)
print("DATASETS =", DATASETS)
print("DEVICE =", DEVICE, "| NUM_GPUS =", NUM_GPUS, "| BATCH_SIZE per GPU =", BATCH_SIZE)
print("WORK_ROOT =", WORK_ROOT)


REPO_GIT_URL = https://github.com/hungnp-dev/RQFormer.git
DATA_ROOT = /kaggle/input/datasets/jks010/rqformer-dataset/data
DATASETS = dior
DEVICE = 0,1 | NUM_GPUS = 2 | BATCH_SIZE per GPU = 2
WORK_ROOT = /kaggle/working/work_dirs


## 2. Khai báo 3 job dataset/config


In [4]:
JOBS = {
    "dior": {
        "title": "RQFormer | DIOR-R | R50 | 3x | Query 500 | t0.85",
        "dataset_label": "DIOR-R",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth",
        "data_link": Path("data/DIOR"),
        "data_input": DATA_INPUTS["dior"],
        "image_source": Path("data/DIOR/JPEGImages-test"),
    },
    "dotav1_0": {
        "title": "RQFormer | DOTA-v1.0 | R50 | 2x | Query 500 | t0.9",
        "dataset_label": "DOTA-v1.0",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.pth",
        "data_link": Path("data/split_ss_dota"),
        "data_input": DATA_INPUTS["dotav1_0"],
        "image_source": Path("data/split_ss_dota/trainval/images"),
    },
    "dotav1_5": {
        "title": "RQFormer | DOTA-v1.5 | R50 | 2x | Query 500 | t0.9",
        "dataset_label": "DOTA-v1.5",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.pth",
        "data_link": Path("data/split_ss_dota1_5"),
        "data_input": DATA_INPUTS["dotav1_5"],
        "image_source": Path("data/split_ss_dota1_5/trainval/images"),
    },
}

def selected_names():
    if DATASETS == "all":
        return list(JOBS)
    names = [x.strip() for x in DATASETS.split(",") if x.strip()]
    unknown = [x for x in names if x not in JOBS]
    if unknown:
        raise ValueError(f"Unknown DATASETS: {unknown}. Valid: {list(JOBS)}")
    return names

print("Selected jobs:", selected_names())


Selected jobs: ['dior']


## 3. Clone source, link data, patch config base

Cell này xử lý lỗi bạn gặp: `mmrotate/.mim/model-index.yml` bằng cách sửa 3 config RQFormer sang dùng base config local:

- `../../../configs/_base_/datasets/dior.py`
- `../../../configs/_base_/datasets/dota.py`
- `../../../configs/_base_/datasets/dotav15.py`
- `../../../configs/_base_/schedules/schedule_3x.py`
- `../../../configs/_base_/default_runtime.py`


In [5]:
import os
import shutil
import subprocess
from pathlib import Path


def get_github_token():
    token = os.environ.get("GITHUB_TOKEN", "")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET_NAME) or ""
    except Exception:
        return ""


def copy_repo_fallback():
    if not REPO_INPUT_FALLBACK.exists():
        raise RuntimeError(
            "Không clone được GitHub và cũng không thấy REPO_INPUT_FALLBACK.\n"
            "Hãy bật Internet/make repo public, thêm Kaggle Secret GITHUB_TOKEN, "
            f"hoặc upload source tại {REPO_INPUT_FALLBACK}"
        )
    ignore = shutil.ignore_patterns(".git", "work_dirs", "__pycache__", "*.pyc", ".ipynb_checkpoints")
    shutil.copytree(REPO_INPUT_FALLBACK, REPO_DIR, ignore=ignore)
    print(f"[OK] Copied repo fallback: {REPO_INPUT_FALLBACK} -> {REPO_DIR}")


def clone_repo():
    if REPO_DIR.exists():
        print(f"[OK] Repo exists: {REPO_DIR}")
        return
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

    public_cmd = ["git", "clone", REPO_GIT_URL, str(REPO_DIR)]
    print("$", " ".join(public_cmd))
    try:
        subprocess.check_call(public_cmd)
        print(f"[OK] Cloned public repo")
        return
    except subprocess.CalledProcessError as exc:
        print(f"[WARN] Public clone failed: {exc.returncode}")

    token = get_github_token()
    if token:
        private_cmd = ["git", "-c", f"http.extraHeader=Authorization: Bearer {token}", "clone", REPO_GIT_URL, str(REPO_DIR)]
        try:
            subprocess.check_call(private_cmd)
            print(f"[OK] Cloned private repo")
            return
        except subprocess.CalledProcessError as exc:
            print(f"[WARN] Private clone failed: {exc.returncode}")

    copy_repo_fallback()


def link_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        print(f"[OK] Exists: {dst}")
        return
    if not src.exists():
        raise FileNotFoundError(f"Không thấy input: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
        print(f"[OK] Symlink: {dst} -> {src}")
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        print(f"[OK] Copied: {src} -> {dst}")


def strip_bom_from_text_files():
    """Remove UTF-8 BOM that breaks Python 3.7 ast.parse on Kaggle."""
    patterns = ["configs/**/*.py", "projects/**/*.py", "tools/**/*.py", "tools/**/*.sh"]
    fixed = []
    for pattern in patterns:
        for path in REPO_DIR.glob(pattern):
            if not path.is_file():
                continue
            raw = path.read_bytes()
            if raw.startswith(b"\xef\xbb\xbf"):
                path.write_bytes(raw[3:])
                fixed.append(path)
    if fixed:
        print("[OK] Removed UTF-8 BOM from:")
        for path in fixed:
            print("  ", path.relative_to(REPO_DIR))
    else:
        print("[OK] No UTF-8 BOM found in config/source files")


def patch_configs_to_local_base():
    replacements = {
        "mmrotate::_base_/datasets/dior.py": "../../../configs/_base_/datasets/dior.py",
        "mmrotate::_base_/datasets/dota.py": "../../../configs/_base_/datasets/dota.py",
        "mmrotate::_base_/datasets/dotav15.py": "../../../configs/_base_/datasets/dotav15.py",
        "mmrotate::_base_/schedules/schedule_3x.py": "../../../configs/_base_/schedules/schedule_3x.py",
        "mmrotate::_base_/default_runtime.py": "../../../configs/_base_/default_runtime.py",
    }
    for name in JOBS:
        cfg = REPO_DIR / JOBS[name]["config"]
        text = cfg.read_text(encoding="utf-8")
        new_text = text
        for old, new in replacements.items():
            new_text = new_text.replace(old, new)
        if new_text != text:
            cfg.write_text(new_text, encoding="utf-8")
            print(f"[OK] Patched base paths: {cfg}")



def patch_visualize_rroi_attention():
    # Patch heatmap visualizer for geometry-aware RRoIAttention.
    path = REPO_DIR / "tools/analysis_tools/visualize_rroi_attention.py"
    if not path.exists():
        print("[SKIP PATCH HEATMAP] Missing", path)
        return
    text = path.read_text(encoding="utf-8")
    if "def forward_with_cache(query, roi_feat, _module=module):" not in text:
        print("[OK] Heatmap visualizer already supports geometry-aware forward")
        return

    old = """        original_forward = module.forward

        def forward_with_cache(query, roi_feat, _module=module):
            bs, num_queries = query.shape[:2]
            attn = _module.attention_weights(query).view(
                bs, num_queries, _module.num_heads,
                _module.roi_pooler_resolution**2)
            attn = attn.softmax(-1)
            _module.last_attention_weights = attn.detach().cpu()

            attn = attn.unsqueeze(-2)
            value = _module.value_proj(
                roi_feat.permute(0, 2, 3, 1)).permute(0, 3, 1,
                                                      2).contiguous()
            value = value.view(bs, num_queries, _module.num_heads, -1,
                               _module.roi_pooler_resolution**2)
            output = (value * attn).sum(-1).view(bs, num_queries,
                                                 _module.embed_dims)
            return _module.output_proj(output)
"""
    new = """        original_forward = module.forward

        def forward_with_cache(query,
                               roi_feat,
                               boxes=None,
                               img_shape=None,
                               _module=module,
                               **kwargs):
            # Mirror RRoIAttention.forward and cache attention weights.
            bs, num_queries = query.shape[:2]
            geometry = None
            if (getattr(_module, 'use_geometry', False) and boxes is not None
                    and boxes.numel() > 0):
                geometry = _module._box_geometry(boxes, img_shape)
                query = query + _module.geometry_embed(geometry).unsqueeze(0)

            attention_logits = _module.attention_weights(query).view(
                bs, num_queries, _module.num_heads,
                _module.roi_pooler_resolution**2)
            if geometry is not None:
                orientation_bias = _module.orientation_bias(geometry).view(
                    1, num_queries, _module.num_heads,
                    _module.roi_pooler_resolution**2)
                attention_logits = attention_logits + orientation_bias

            attention_weights = attention_logits.softmax(-1)
            _module.last_attention_weights = attention_weights.detach().cpu()

            attn = attention_weights.unsqueeze(-2)
            value = _module.value_proj(
                roi_feat.permute(0, 2, 3, 1)).permute(0, 3, 1,
                                                      2).contiguous()
            value = value.view(bs, num_queries, _module.num_heads, -1,
                               _module.roi_pooler_resolution**2)
            output = (value * attn).sum(-1).view(bs, num_queries,
                                                 _module.embed_dims)
            return _module.output_proj(output)
"""
    if old not in text:
        print("[WARN PATCH HEATMAP] Expected old forward_with_cache block not found")
        return
    path.write_text(text.replace(old, new), encoding="utf-8")
    print("[OK] Patched heatmap visualizer for boxes/img_shape")


def prepare_data_and_ckpts():
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"Không thấy DATA_ROOT: {DATA_ROOT}")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    for name in selected_names():
        job = JOBS[name]
        link_or_copy(job["data_input"], REPO_DIR / job["data_link"])
        if CKPT_INPUT is None:
            print("[SKIP CKPT INPUT] Không dùng checkpoint ngoài; train từ đầu hoặc resume từ work_dirs.")
            continue
        ckpt_src = CKPT_INPUT / job["ckpt"]
        ckpt_dst = CKPT_DIR / job["ckpt"]
        if ckpt_src.exists():
            link_or_copy(ckpt_src, ckpt_dst)
        else:
            print(f"[WARN] Không thấy checkpoint gốc: {ckpt_src}; train từ đầu vẫn chạy được.")

clone_repo()
os.chdir(REPO_DIR)
strip_bom_from_text_files()
patch_configs_to_local_base()
patch_visualize_rroi_attention()
prepare_data_and_ckpts()
print("cwd =", Path.cwd())


[OK] Repo exists: /kaggle/working/RQFormer
[OK] No UTF-8 BOM found in config/source files
[OK] Exists: /kaggle/working/RQFormer/data/DIOR
[SKIP CKPT INPUT] Không dùng checkpoint ngoài; train từ đầu hoặc resume từ work_dirs.
cwd = /kaggle/working/RQFormer


## 4. Setup môi trường

Không dùng shell script. Cell này cài trực tiếp các dependency cần thiết rồi `pip install -e .`.


In [6]:
import importlib.util
import sys


def has_module(name):
    return importlib.util.find_spec(name) is not None


def run(cmd, cwd=REPO_DIR, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd, env=env)


def env_ready():
    try:
        import torch, mmcv, mmdet, mmengine, mmrotate
        print("Environment ready:", torch.__version__, mmcv.__version__, mmdet.__version__, mmrotate.__version__)
        return True
    except Exception as exc:
        print("Environment not ready:", repr(exc))
        return False

if RUN_SETUP:
    if not env_ready():
        run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
        req_build = REPO_DIR / "requirements/build.txt"
        if req_build.exists():
            run([sys.executable, "-m", "pip", "install", "-r", str(req_build)])
        run([sys.executable, "-m", "pip", "install", "packaging", "matplotlib", "pycocotools", "six", "terminaltables", "scipy", "scikit-learn", "imagecorruptions"])
        if not has_module("openmim"):
            run([sys.executable, "-m", "pip", "install", "-U", "openmim"])
        if not has_module("mmcv"):
            run([sys.executable, "-m", "mim", "install", "mmcv>=2.0.0rc2,<2.1.0"])
        if not has_module("mmengine"):
            run([sys.executable, "-m", "pip", "install", "mmengine>=0.1.0"])
        if not has_module("mmdet"):
            run([sys.executable, "-m", "pip", "install", "mmdet>=3.0.0rc5,<3.1.0"])
        run([sys.executable, "-m", "pip", "install", "-v", "-e", "."])
    env_ready()
else:
    print("[SKIP SETUP]")


Environment ready: 1.13.0 2.0.1 3.0.0 1.0.0rc1
Environment ready: 1.13.0 2.0.1 3.0.0 1.0.0rc1


## 5. Mapping source, cải tiến và notebook

Notebook này chạy trực tiếp các file Python gốc, không tự viết lại kiến trúc model. Các điểm dưới đây là mapping bắt buộc để kiểm tra notebook có bám đúng source hiện tại hay không.

| Thành phần | File Python/config | Notebook mirror | Mục đích |
|---|---|---|---|
| Geometry-aware RRoI attention | `projects/RQFormer/rroiformer/rroiattention.py` -> `RRoIAttention(use_geometry=True)`, `_box_geometry`, `geometry_embed`, `orientation_bias` | Các config trong `JOBS[*]["config"]` trỏ đúng 3 config đã bật `rroi_attn_cfg.use_geometry=True` | Bổ sung đặc trưng hình học hộp xoay vào query và attention logits |
| Pairwise query geometry relation | `projects/RQFormer/rroiformer/rroiformer_decoder_layer.py` -> `RRoIFormerDecoderLayer(use_geometry_relation=True)`, `_relation_geometry`, `geometry_relation_scale` | Notebook không sửa layer; chỉ gọi `tools/train.py`/`tools/test.py` với config đã bật `use_geometry_relation=True` | Cho query trao đổi ngữ cảnh hình học; `geometry_relation_scale` khởi tạo 0 để ổn định fine-tuning |
| Bật cải tiến cho 3 dataset chính | `projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py`, `...dotav1.0.py`, `...dotav1.5.py` | `JOBS` mirror đúng 3 config, dataset path và tên checkpoint | Đảm bảo train/test đúng biến thể RQFormer-2 hiện tại |
| Checkpoint một file duy nhất | `tools/run_all_rqformer_visualize.sh` -> `latest_ckpt`, `keep_single_ckpt`, `resolve_ckpt`, `run_train` | `latest_ckpt`, `keep_single_ckpt`, `resolve_ckpt`, `run_train_job` trong notebook | Train xong chỉ giữ 1 `.pth` theo format từng dataset, ví dụ `rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth` |
| Resume và batch 3090 Ti | Script dùng `BATCH_SIZE=2`, `--resume`, `default_hooks.checkpoint.max_keep_ckpts=1` | Notebook dùng cùng cơ chế resume/checkpoint; với Kaggle dual T4 dùng `NUM_GPUS=2`, `BATCH_SIZE=1` mỗi GPU | Chạy ổn trên T4 15GB x2 hoặc đổi về `NUM_GPUS=1`, `BATCH_SIZE=2` cho RTX 3090 Ti |
| Dual GPU Kaggle | `tools/train.py` và `tools/test.py` hỗ trợ `--launcher pytorch` | Notebook dùng `python -m torch.distributed.run --nproc_per_node=NUM_GPUS ... --launcher pytorch` khi `NUM_GPUS > 1` | DDP đúng chuẩn, mỗi GPU một process, global batch = `BATCH_SIZE * NUM_GPUS` |

Công thức cải tiến được hiện thực trong source:

- Geometry injection trong `RRoIAttention.forward`: `q'_i = q_i + g(b_i)`, với `b_i = (x_i, y_i, w_i, h_i, theta_i)` là hộp xoay chuẩn hóa, `g(.)` là `geometry_embed`.
- Orientation-aware attention: `A_i = softmax(W_q q'_i + o(b_i))`, với `o(.)` là `orientation_bias` sinh bias theo hình học/góc cho từng head và grid trong RRoI.
- Pairwise relation trong decoder layer: `q'_i = q_i + alpha * Dropout(sum_j softmax(phi(r_ij)) q_j)`, với `r_ij` là quan hệ tương đối giữa hai query box, `phi(.)` là `geometry_relation`, và `alpha` là `geometry_relation_scale` học được, khởi tạo bằng 0.


## 6. Hàm chạy train/test/analysis trực tiếp

Các hàm dưới đây mirror logic quan trọng của `tools/run_all_rqformer_visualize.sh`: resume checkpoint dở, giữ đúng 1 checkpoint theo tên chuẩn, test bằng checkpoint vừa train, rồi chạy chart/confusion/bbox/heatmap nếu bật stage.


In [7]:

def use_distributed():
    return int(NUM_GPUS) > 1


def ddp_cmd(script, *args):
    if use_distributed():
        return [
            sys.executable, "-m", "torch.distributed.run",
            "--nproc_per_node", str(NUM_GPUS),
            "--master_port", str(MASTER_PORT),
            script, *map(str, args),
            "--launcher", "pytorch",
        ]
    return [sys.executable, script, *map(str, args)]


def per_gpu_note():
    global_batch = int(BATCH_SIZE) * max(1, int(NUM_GPUS))
    print(f"[GPU CONFIG] CUDA_VISIBLE_DEVICES={DEVICE} | NUM_GPUS={NUM_GPUS} | batch/GPU={BATCH_SIZE} | global batch={global_batch}")


def ckpt_name_for(job):
    return job["ckpt"]


def latest_ckpt(path: Path, ckpt_name=None):
    path = Path(path)
    if not path.exists():
        return None
    if ckpt_name:
        named = path / ckpt_name
        if named.exists():
            return named
    latest = path / "latest.pth"
    if latest.exists():
        return latest
    ckpts = sorted(
        path.glob("epoch_*.pth"),
        key=lambda p: int(p.stem.split("_")[-1]) if p.stem.split("_")[-1].isdigit() else -1,
    )
    return ckpts[-1] if ckpts else None


def keep_single_ckpt(path: Path, ckpt_name: str):
    path = Path(path)
    src = latest_ckpt(path, ckpt_name)
    if src is None or not src.exists():
        print("[WARN NO TRAIN CKPT]", path)
        return None
    final = path / ckpt_name
    if src.resolve() != final.resolve():
        tmp = final.with_suffix(final.suffix + ".tmp")
        shutil.copy2(src, tmp)
        tmp.replace(final)
    for p in path.glob("*.pth"):
        if p.name != ckpt_name:
            p.unlink()
    print("[SINGLE CKPT]", final)
    return final


def count_files(path: Path):
    return sum(1 for p in path.rglob("*") if p.is_file()) if path.exists() else 0


def cfg_options(train=True):
    opts = [
        f"train_dataloader.batch_size={BATCH_SIZE}",
        f"train_dataloader.num_workers={NUM_WORKERS}",
        f"val_dataloader.batch_size={BATCH_SIZE}",
        f"val_dataloader.num_workers={NUM_WORKERS}",
        f"test_dataloader.batch_size={BATCH_SIZE}",
        f"test_dataloader.num_workers={NUM_WORKERS}",
        "default_hooks.checkpoint.max_keep_ckpts=1",
    ]
    if MAX_EPOCHS is not None:
        opts.append(f"train_cfg.max_epochs={MAX_EPOCHS}")
    if LR is not None:
        opts.append(f"optim_wrapper.optimizer.lr={LR}")
    return opts


def run_train_job(name, job, root):
    train_dir = root / "train"
    done = root / ".train.done"
    ckpt_name = ckpt_name_for(job)

    if not RUN_TRAIN:
        print("[SKIP TRAIN]", name)
        return latest_ckpt(train_dir, ckpt_name)

    ckpt = latest_ckpt(train_dir, ckpt_name)
    if done.exists() and ckpt and not FORCE:
        print("[SKIP TRAIN DONE]", name, ckpt)
        return keep_single_ckpt(train_dir, ckpt_name) or ckpt

    train_dir.mkdir(parents=True, exist_ok=True)
    cmd = ddp_cmd("tools/train.py", job["config"], "--work-dir", str(train_dir))

    if ckpt and not FORCE:
        print("[RESUME TRAIN]", job["title"], ckpt)
        cmd.append("--resume")
    else:
        print("[RUN TRAIN]", job["title"])

    cmd += ["--cfg-options", *cfg_options(train=True)]
    run(cmd)
    final = keep_single_ckpt(train_dir, ckpt_name)
    done.touch()
    return final


def resolve_ckpt(name, job, root):
    trained = latest_ckpt(root / "train", ckpt_name_for(job))
    if trained:
        return trained
    if CKPT_INPUT is None:
        return None
    external = CKPT_DIR / job["ckpt"]
    return external if external.exists() else None


def run_test_job(name, job, root, ckpt):
    test_dir = root / "test"
    pred = test_dir / "predictions.pkl"
    done = root / ".test.done"
    if not RUN_TEST:
        print("[SKIP TEST]", name)
        return pred
    if done.exists() and pred.exists() and not FORCE:
        print("[SKIP TEST DONE]", name)
        return pred
    if not ckpt or not Path(ckpt).exists():
        print("[SKIP TEST NO CKPT]", name)
        return pred
    test_dir.mkdir(parents=True, exist_ok=True)
    cmd = ddp_cmd(
        "tools/test.py", job["config"], str(ckpt),
        "--work-dir", str(test_dir), "--out", str(pred),
    ) + [
        "--cfg-options",
        f"test_dataloader.batch_size={BATCH_SIZE}",
        f"test_dataloader.num_workers={NUM_WORKERS}",
    ]
    print("[RUN TEST]", job["title"])
    run(cmd)
    done.touch()
    return pred


def run_optional_analysis(name, job, root, ckpt, pred):
    chart_dir = root / "charts"
    bbox_dir = root / "images" / "bboxes"
    heatmap_dir = root / "images" / "heatmaps"
    image_source = REPO_DIR / job["image_source"]

    if RUN_CHARTS and not (root / ".charts.done").exists():
        plot_script = REPO_DIR / "tools/analysis_tools/plot_rqformer_charts.py"
        logs = list((root / "train").rglob("*.json")) + list((root / "test").rglob("*.json"))
        if plot_script.exists() and logs:
            chart_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(plot_script), *map(str, logs), "--out-dir", str(chart_dir), "--title", job["title"]])
            (root / ".charts.done").touch()
        else:
            print("[SKIP CHARTS] missing script or json logs")

    if RUN_CONFUSION and not (root / ".confusion.done").exists():
        cm_script = REPO_DIR / "tools/analysis_tools/confusion_matrix.py"
        if cm_script.exists() and pred.exists():
            out_dir = chart_dir / "confusion_matrix"
            out_dir.mkdir(parents=True, exist_ok=True)
            run([
                sys.executable, str(cm_script), job["config"], str(pred), str(out_dir),
                "--score-thr", str(SCORE_THR),
                "--cfg-options",
                f"test_dataloader.batch_size={BATCH_SIZE}",
                f"test_dataloader.num_workers={NUM_WORKERS}",
            ])
            (root / ".confusion.done").touch()
        else:
            print("[SKIP CONFUSION] missing script or predictions")

    if RUN_BBOX and not (root / ".bbox.done").exists():
        bbox_script = REPO_DIR / "tools/analysis_tools/visualize_rqformer_bboxes.py"
        if bbox_script.exists() and ckpt and Path(ckpt).exists() and image_source.exists():
            bbox_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(bbox_script), job["config"], str(ckpt), str(image_source), "--out-dir", str(bbox_dir), "--device", "cuda:0", "--score-thr", str(SCORE_THR), "--max-images", str(SAMPLE_IMAGES)])
            (root / ".bbox.done").touch()
        else:
            print("[SKIP BBOX] missing script, ckpt, or image source")

    if RUN_HEATMAP and not (root / ".heatmap.done").exists():
        heat_script = REPO_DIR / "tools/analysis_tools/visualize_rroi_attention.py"
        if heat_script.exists() and ckpt and Path(ckpt).exists() and image_source.exists():
            heatmap_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(heat_script), job["config"], str(ckpt), str(image_source), "--out-dir", str(heatmap_dir), "--device", "cuda:0", "--score-thr", str(SCORE_THR), "--max-images", str(SAMPLE_IMAGES)])
            (root / ".heatmap.done").touch()
        else:
            print("[SKIP HEATMAP] missing script, ckpt, or image source")

    print("Charts:", count_files(chart_dir), "BBoxes:", count_files(bbox_dir), "Heatmaps:", count_files(heatmap_dir))


In [8]:
import shutil
from pathlib import Path

DATASETS = "dior"

RUN_SETUP = 0
RUN_TRAIN = 0
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
RUN_SUMMARY = 1

FORCE = 1
SAMPLE_IMAGES = 10
SCORE_THR = 0.3

epoch_ckpt = WORK_ROOT / "rroiformer" / "dior" / "train" / "epoch_4.pth"
named_ckpt = WORK_ROOT / "rroiformer" / "dior" / "train" / "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth"

print("Check:", epoch_ckpt)
assert epoch_ckpt.exists(), f"Không thấy checkpoint: {epoch_ckpt}"

shutil.copy2(epoch_ckpt, named_ckpt)
print("Use checkpoint:", named_ckpt)

Check: /kaggle/working/work_dirs/rroiformer/dior/train/epoch_4.pth
Use checkpoint: /kaggle/working/work_dirs/rroiformer/dior/train/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth


## 7. Chạy toàn bộ workflow trực tiếp bằng Python

Cell này chạy từng dataset đã chọn theo `DATASETS`, dùng `tools/train.py`, `tools/test.py` và các script analysis/visualization. Không gọi `tools/run_all_rqformer_visualize.sh`, nhưng logic checkpoint/resume/batch được mirror theo script hiện tại.


In [9]:
os.environ["CUDA_VISIBLE_DEVICES"] = str(DEVICE)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
per_gpu_note()

for name in selected_names():
    job = JOBS[name]
    root = WORK_ROOT / "rroiformer" / name
    (root / "logs").mkdir(parents=True, exist_ok=True)
    print("=" * 80)
    print("[JOB]", job["title"])
    print("Output:", root)

    cfg = REPO_DIR / job["config"]
    data = REPO_DIR / job["data_link"]
    if not cfg.exists():
        raise FileNotFoundError(cfg)
    if not data.exists():
        raise FileNotFoundError(data)

    trained = run_train_job(name, job, root)
    ckpt = resolve_ckpt(name, job, root)
    print("[USING CKPT]", ckpt)
    pred = run_test_job(name, job, root, ckpt)
    run_optional_analysis(name, job, root, ckpt, pred)


[GPU CONFIG] CUDA_VISIBLE_DEVICES=0,1 | NUM_GPUS=2 | batch/GPU=2 | global batch=4
[JOB] RQFormer | DIOR-R | R50 | 3x | Query 500 | t0.85
Output: /kaggle/working/work_dirs/rroiformer/dior
[SKIP TRAIN] dior
[USING CKPT] /kaggle/working/work_dirs/rroiformer/dior/train/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth
[RUN TEST] RQFormer | DIOR-R | R50 | 3x | Query 500 | t0.85
$ /opt/conda/bin/python3.7 -m torch.distributed.run --nproc_per_node 2 --master_port 29500 tools/test.py projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py /kaggle/working/work_dirs/rroiformer/dior/train/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth --work-dir /kaggle/working/work_dirs/rroiformer/dior/test --out /kaggle/working/work_dirs/rroiformer/dior/test/predictions.pkl --launcher pytorch --cfg-options test_dataloader.batch_size=2 test_dataloader.num_workers=2


/opt/conda/lib/python3.7/site-packages/mmengine/utils/dl_utils/setup_env.py:57: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  'Setting MKL_NUM_THREADS environment variable for each process'
/opt/conda/lib/python3.7/site-packages/mmengine/utils/dl_utils/setup_env.py:57: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  'Setting MKL_NUM_THREADS environment variable for each process'


07/14 07:52:47 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.7.12 | packaged by conda-forge | (default, Oct 26 2021, 06:08:53) [GCC 9.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1126039877
    GPU 0,1: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 11.3, V11.3.109
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.1) 9.4.0
    PyTorch: 1.13.0
    PyTorch compiling details: PyTorch built with:
  - GCC 9.4
  - C++ Version: 201402
  - Intel(R) oneAPI Math Kernel Library Version 2023.0-Product Build 20221128 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.6.0 (Git Hash 52b5f107dd9cf10910aaa19cb47f3abf9b349815)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.3
  - NVCC architecture flags: -gencode;arch=compute_3

/opt/conda/lib/python3.7/site-packages/mmdet/structures/bbox/base_boxes.py:62: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /usr/local/src/pytorch/torch/csrc/utils/tensor_new.cpp:230.)
  data = torch.as_tensor(data)
/opt/conda/lib/python3.7/site-packages/mmdet/structures/bbox/base_boxes.py:62: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /usr/local/src/pytorch/torch/csrc/utils/tensor_new.cpp:230.)
  data = torch.as_tensor(data)
/opt/conda/lib/python3.7/site-packages/mmdet/structures/bbox/base_boxes.py:62: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray wit

07/14 07:53:36 - mmengine - INFO - Epoch(test) [  50/2935]    eta: 0:17:52  time: 0.3718  data_time: 0.0103  memory: 746  
07/14 07:53:52 - mmengine - INFO - Epoch(test) [ 100/2935]    eta: 0:16:34  time: 0.3296  data_time: 0.0067  memory: 746  
07/14 07:54:09 - mmengine - INFO - Epoch(test) [ 150/2935]    eta: 0:16:00  time: 0.3338  data_time: 0.0064  memory: 746  
07/14 07:54:26 - mmengine - INFO - Epoch(test) [ 200/2935]    eta: 0:15:38  time: 0.3381  data_time: 0.0063  memory: 746  
07/14 07:54:43 - mmengine - INFO - Epoch(test) [ 250/2935]    eta: 0:15:22  time: 0.3452  data_time: 0.0065  memory: 746  
07/14 07:55:01 - mmengine - INFO - Epoch(test) [ 300/2935]    eta: 0:15:08  time: 0.3513  data_time: 0.0061  memory: 745  
07/14 07:55:18 - mmengine - INFO - Epoch(test) [ 350/2935]    eta: 0:14:54  time: 0.3532  data_time: 0.0059  memory: 746  
07/14 07:55:35 - mmengine - INFO - Epoch(test) [ 400/2935]    eta: 0:14:36  time: 0.3442  data_time: 0.0061  memory: 746  
07/14 07:55:52 -

/opt/conda/lib/python3.7/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /usr/local/src/pytorch/aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11726_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11727_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11728_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11729_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11730_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11731_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11732_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11733_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11734_bbox.jpg
[SAVED] /kaggle/working/work_dirs/rroiformer/dior/images/bboxes/11735_bbox.jpg
$ /opt/conda/bin/python3.7 /kaggle/working/RQFormer/tools/analysis_tools/visualize_rroi_attention.py projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py /kaggle/working/work_dirs/

/opt/conda/lib/python3.7/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /usr/local/src/pytorch/aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Traceback (most recent call last):
  File "/kaggle/working/RQFormer/tools/analysis_tools/visualize_rroi_attention.py", line 188, in <module>
    main()
  File "/kaggle/working/RQFormer/tools/analysis_tools/visualize_rroi_attention.py", line 172, in main
    result = inference_detector(model, img_path)
  File "/opt/conda/lib/python3.7/site-packages/mmdet/apis/inference.py", line 177, in inference_detector
    results = model.test_step(data_)[0]
  File "/opt/conda/lib/python3.7/site-packages/mmengine/model/base_model/base_model.py", line 145, in test_step
    return self._run_forward(data, mode='predict')  # type: ignore
  File "/opt/conda/lib/python3.7/site-pack

CalledProcessError: Command '['/opt/conda/bin/python3.7', '/kaggle/working/RQFormer/tools/analysis_tools/visualize_rroi_attention.py', 'projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py', '/kaggle/working/work_dirs/rroiformer/dior/train/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth', '/kaggle/working/RQFormer/data/DIOR/JPEGImages-test', '--out-dir', '/kaggle/working/work_dirs/rroiformer/dior/images/heatmaps', '--device', 'cuda:0', '--score-thr', '0.3', '--max-images', '10']' returned non-zero exit status 1.

## 8. Ghi summary results

Summary chỉ lấy metric có trong log/evaluator; cột nào không có thì để `-`, không tự bịa thêm chỉ số.


In [ ]:
summary_script = REPO_DIR / "tools/analysis_tools/write_rqformer_summary.py"
summary = WORK_ROOT / "rroiformer" / "summary_results.md"
if RUN_SUMMARY and summary_script.exists():
    run([sys.executable, str(summary_script), "--repo-dir", str(REPO_DIR), "--work-root", str(WORK_ROOT), "--out", str(summary)])
    print(summary.read_text(encoding="utf-8", errors="replace"))
else:
    print("[SKIP SUMMARY] missing script or RUN_SUMMARY=0")


## 9. Show toàn bộ kết quả và visualization trong notebook

Cell này in checkpoint/prediction/log/summary ra output notebook, đồng thời display toàn bộ chart, bbox visualization và heatmap đã sinh. File vẫn được giữ trong `WORK_ROOT` để zip/download ở bước sau.


In [ ]:
from pathlib import Path

try:
    from IPython.display import display, Image as IPyImage, Markdown
except Exception:
    display = None
    IPyImage = None
    Markdown = None

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
TEXT_EXTS = {".log", ".txt", ".md", ".json"}


def list_files(path: Path, exts=None):
    if not path.exists():
        return []
    files = [p for p in path.rglob("*") if p.is_file()]
    if exts is not None:
        files = [p for p in files if p.suffix.lower() in exts]
    return sorted(files)


def print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def tail_text(path: Path, n=40):
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        return "\n".join(lines[-n:])
    except Exception as exc:
        return f"[cannot read {path}: {exc}]"


def display_images(title, files):
    print_section(title)
    if not files:
        print("[NONE]")
        return
    for path in files:
        print(path)
        if display is not None and IPyImage is not None:
            try:
                display(Markdown(f"**{path.relative_to(WORK_ROOT)}**"))
                display(IPyImage(filename=str(path)))
            except Exception as exc:
                print(f"[DISPLAY SKIP] {path}: {exc}")


def show_job_outputs(name):
    root = WORK_ROOT / "rroiformer" / name
    print_section(f"JOB OUTPUT: {name}")
    print("Root:", root)
    if not root.exists():
        print("[MISSING ROOT]")
        return

    checkpoints = list_files(root / "train", {".pth"})
    predictions = list_files(root / "test", {".pkl"})
    logs = list_files(root, TEXT_EXTS)
    charts = list_files(root / "charts", IMAGE_EXTS)
    bboxes = list_files(root / "images" / "bboxes", IMAGE_EXTS)
    heatmaps = list_files(root / "images" / "heatmaps", IMAGE_EXTS)

    print("Checkpoints:")
    for f in checkpoints:
        print(" ", f)
    print("Predictions:")
    for f in predictions:
        print(" ", f)
    print("Logs/Text/JSON:")
    for f in logs:
        print(" ", f)
    print(f"Charts: {len(charts)} | BBoxes: {len(bboxes)} | Heatmaps: {len(heatmaps)}")

    readable_logs = [p for p in logs if p.suffix.lower() in {".log", ".txt", ".md"}]
    if readable_logs:
        newest = max(readable_logs, key=lambda p: p.stat().st_mtime)
        print_section(f"TAIL LOG: {newest}")
        print(tail_text(newest, n=40))

    display_images(f"CHARTS: {name}", charts)
    display_images(f"BBOX VISUALIZATION: {name}", bboxes)
    display_images(f"HEATMAP VISUALIZATION: {name}", heatmaps)


def show_all_workflow_outputs():
    for name in selected_names():
        show_job_outputs(name)

    summary = WORK_ROOT / "rroiformer" / "summary_results.md"
    if summary.exists():
        print_section("SUMMARY RESULTS")
        text = summary.read_text(encoding="utf-8", errors="replace")
        print(text)
        if display is not None and Markdown is not None:
            display(Markdown(text))

show_all_workflow_outputs()


## 10. Xem cây output và zip kết quả

Zip này chứa checkpoint duy nhất theo tên chuẩn, logs, predictions, charts, bbox/heatmap và `summary_results.md` để tải về hoặc resume lần sau.


In [ ]:
def tree(path: Path, max_depth=4):
    if not path.exists():
        print("Not found:", path)
        return
    base_depth = len(path.parts)
    for p in sorted(path.rglob("*")):
        depth = len(p.parts) - base_depth
        if depth > max_depth:
            continue
        indent = "  " * max(depth - 1, 0)
        print(f"{indent}{p.name}{'/' if p.is_dir() else ''}")

print("Output root:", WORK_ROOT)
tree(WORK_ROOT / "rroiformer", max_depth=4)

archive_base = Path("/kaggle/working/rqformer2_full_workflow_outputs")
zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(WORK_ROOT))
print("Saved zip:", zip_path)


## 11. Chạy nhanh

Mặc định chỉ cần chỉnh cell cấu hình đầu tiên rồi bấm **Run All**.

Chạy riêng DIOR-R trên Kaggle dual T4:

```python
DATASETS = "dior"
DEVICE = "0,1"
NUM_GPUS = 2
BATCH_SIZE = 1      # batch mỗi GPU, global batch = 2
MAX_EPOCHS = 1      # bỏ None để chạy đúng 36 epoch theo config 3x
RUN_SETUP = 1
RUN_TRAIN = 1
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
RUN_SUMMARY = 1
```

Chạy single GPU/RTX 3090 Ti thì đổi:

```python
DEVICE = "0"
NUM_GPUS = 1
BATCH_SIZE = 2
```

Chỉ train DOTA-v1.0:

```python
DATASETS = "dotav1_0"
RUN_SETUP = 1
RUN_TRAIN = 1
RUN_TEST = 0
RUN_CHARTS = 0
RUN_CONFUSION = 0
RUN_BBOX = 0
RUN_HEATMAP = 0
RUN_SUMMARY = 0
```

Chỉ test/visualize từ checkpoint đã train trong `work_dirs`:

```python
DATASETS = "dior"
RUN_SETUP = 0
RUN_TRAIN = 0
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
RUN_SUMMARY = 1
```

Chạy lại toàn bộ một dataset:

```python
DATASETS = "dior"
FORCE = 1
```
